### Loading raw data (skip for now)

In [2]:
import os
import glob
import time
import polars as pl

# 1. Direct path setup
RAW_DIR = "/media/storage0/allison/BatchSystem-2025"

# Find all 32 partition files directly in RAW_DIR
valid_paths = sorted(glob.glob(os.path.join(RAW_DIR, "fife_raw_part_*.parquet")))
print(f"Found {len(valid_paths)} Parquet partition files.")
assert valid_paths, f"No files found in {RAW_DIR} matching 'fife_raw_part_*.parquet'!"

# Calculate dataset size on disk
total_gb = sum(os.path.getsize(p) for p in valid_paths) / 1e9
print(f"Total dataset size on disk: {total_gb:.2f} GB across {len(valid_paths)} files.")

# 2. Columns to extract
NEED = [
    "ExitCode","ExitSignal","JobStatus","RemoveReason","LastHoldReason","LastHoldReasonCode",
    "NumJobStarts","QDate","JobStartDate","JobCurrentStartDate","JobCurrentStartExecutingDate",
    "CompletionDate","RemoteWallClockTime","x509UserProxyExpiration","Owner","AccountingGroup",
    "Group","POMS4_CAMPAIGN_ID","POMS4_CAMPAIGN_NAME","POMS4_CAMPAIGN_STAGE_NAME",
    "POMS4_CAMPAIGN_STAGE_ID","POMS4_CAMPAIGN_TYPE","POMS4_TEST_LAUNCH","Jobsub_Group",
    "SingularityImage","Blacklist_Sites","RequestCpus","RequestDisk","RequestMemory","RequestSlots",
    "CpusProvisioned","DiskProvisioned","MemoryProvisioned","ExecutableSize","TransferInputSizeMB",
    "JOB_EXPECTED_MAX_LIFETIME","TotalSubmitProcs","MATCH_EXP_JOB_GLIDEIN_Site","MATCH_GLIDEIN_Entry_Name",
    "MATCH_GLIDEIN_SiteWMS_Queue","MachineAttrGLIDEIN_ResourceName0","MachineAttrCpus0","LastRemoteHost",
    "MATCH_EXP_JOB_Site","ClusterId",
]

# Quick schema check from the first partition (instant, no data read)
schema_sample = pl.read_parquet_schema(valid_paths[0])
have = [c for c in NEED if c in schema_sample.names()]
missing = [c for c in NEED if c not in schema_sample.names()]
if missing:
    print("Columns in NEED not present in raw dataset:", missing)

# 3. Create LazyFrame across all 32 partitions
try:
    lf = pl.scan_parquet(valid_paths, missing_columns="insert")
except TypeError:
    lf = pl.scan_parquet(valid_paths, allow_missing_columns=True)

# Select only needed columns (Projection Pushdown saves massive amounts of RAM/CPU)
lf = lf.select(have)

if "AccountingGroup" in have:
    lf = lf.with_columns(pl.col("AccountingGroup").str.split(".").list.first().alias("Group"))
    if "Group" not in have:
        have.append("Group")

schema = lf.collect_schema()

# Timestamp normalization helper
def to_sec(col):
    if isinstance(schema[col], pl.Datetime):
        return pl.col(col).dt.epoch(time_unit="s").cast(pl.Float64)
    v = pl.col(col).cast(pl.Float64, strict=False)
    return (pl.when(v > 1e17).then(v / 1e9)
              .when(v > 1e14).then(v / 1e6)
              .when(v > 1e11).then(v / 1e3)
              .otherwise(v))

qcol = next((c for c in ["QDate_ms","QDate"] if c in have), None)
scol = next((c for c in ["JobStartDate_ms","JobStartDate","JobCurrentStartDate_ms","JobCurrentStartExecutingDate"] if c in have), None)

# Labeling & Feature Logic
EXIT_SUB=[9,65,90,91,124,126,127,130,131,137]; EXIT_HW=[129,143]
SIG_SUB=[2,3,9]; SIG_HW=[1,15]; HOLD_APP=[3,6,16]; HOLD_SUB=[1,4,7,8,12,13,26,32,33,34,35]

ec, es, js = pl.col("ExitCode"), pl.col("ExitSignal"), pl.col("JobStatus")
lhr = pl.col("LastHoldReasonCode") if "LastHoldReasonCode" in have else pl.lit(None, dtype=pl.Int64)
rr  = pl.col("RemoveReason").fill_null("") if "RemoveReason" in have else pl.lit("")

failed = ((ec.is_not_null() & (ec != 0)) | es.is_not_null() | (js == 3)).cast(pl.Int8)
ftype = (pl.when((ec == 0) & es.is_null() & (js != 3)).then(-1)
   .when(es.is_in(SIG_SUB)).then(2).when(es.is_in(SIG_HW)).then(1).when(es.is_not_null()).then(0)
   .when(ec.is_in(EXIT_SUB)).then(2).when(ec.is_in(EXIT_HW)).then(1).when(ec.is_not_null() & (ec != 0)).then(0)
   .when(lhr.is_in(HOLD_APP)).then(0).when(lhr.is_in(HOLD_SUB)).then(2)
   .when(rr.str.contains("OtherJobRemoveRequirements") | rr.str.contains("Node Error: DAG node")).then(0)
   .when(rr.str.contains("exceeding job limits") | rr.str.contains("opportunistic")).then(2)
   .when(rr.str.contains("Held 14 days") | rr.str.contains("PeriodicRemove")).then(0)
   .when(rr.str.contains("JobRouter")).then(1)
   .when(rr.str.contains(r"via condor_rm \(by user")).then(-1)
   .when(js == 3).then(1).otherwise(-1)).cast(pl.Int8)

wait = (to_sec(scol) - to_sec(qcol)) if (scol and qcol) else pl.lit(None, dtype=pl.Float64)

lf = lf.with_columns([
    failed.alias("Failed"), 
    ftype.alias("fault_type"),
    pl.when(wait > 0).then(wait).otherwise(None).alias("wait_s")
]).with_columns((pl.col("fault_type") == 1).cast(pl.Int8).alias("hw_fault"))

# Filter stragglers (jobs queued before the data window (2025-02-01 00:00 UTC))
QMIN_S = 1738368000
if qcol:
    lf = lf.filter(to_sec(qcol) >= QMIN_S)

print("Lazy execution plan initialized successfully!")

Found 32 Parquet partition files.
Total dataset size on disk: 1.77 GB across 32 files.
Lazy execution plan initialized successfully!


### Reloading data

In [1]:
%%time
import os
import json
import numpy as np

SAVE_DIR = "/mnt/scratch/fast0/amaustin/datasets/fife/"
targets = np.load(os.path.join(SAVE_DIR, "targets_and_masks.npz"))

Xmatch = np.load(os.path.join(SAVE_DIR, "Xmatch.npy"), mmap_mode="r")
Xsub = np.load(os.path.join(SAVE_DIR, "Xsub.npy"), mmap_mode="r")

failed = np.load(os.path.join(SAVE_DIR, "failed.npy"), mmap_mode="r")
hw = np.load(os.path.join(SAVE_DIR, "hw.npy"), mmap_mode="r")
wait_sv = np.load(os.path.join(SAVE_DIR, "wait_sv.npy"), mmap_mode="r")
tr_mask = np.load(os.path.join(SAVE_DIR, "tr_mask.npy"), mmap_mode="r")
te_mask = np.load(os.path.join(SAVE_DIR, "te_mask.npy"), mmap_mode="r")

job_starts = targets["jst"]

with open(os.path.join(SAVE_DIR, "schema_meta.json"), "r") as f:
    meta = json.load(f)

XMATCH_COLS = meta["XMATCH_COLS"]
XSUB_COLS = meta["XSUB_COLS"]
cards = meta["cards"]
NCAT_MATCH = meta["NCAT_MATCH"]
NCAT_SUB = meta["NCAT_SUB"]
n_base = meta["n_base"]

X = Xmatch[:, :n_base]

print(f"Loaded Xmatch {Xmatch.shape} and Xsub {Xsub.shape}")
print(f"Train split: {tr_mask.sum():,} rows | Test split: {te_mask.sum():,} rows")

Loaded Xmatch (64025075, 46) and Xsub (64025075, 27)
Train split: 48,546,633 rows | Test split: 15,478,442 rows
CPU times: user 3.77 s, sys: 97.3 ms, total: 3.87 s
Wall time: 1.17 s


### Interactive Data Explorer

In [2]:
import importlib
import vis.dr
import vis.app

importlib.reload(vis.dr)
importlib.reload(vis.app)
from vis.app import build_explorer

app_layout = build_explorer(
    Xmatch=Xmatch,
    XMATCH_COLS=XMATCH_COLS,
    failed=failed,
    hw=hw,
    job_starts=job_starts,
    split_mask=tr_mask,
)
display(app_layout)

/home/amaustin/miniconda3/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/amaustin/miniconda3/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
